In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, classification_report, 
                             confusion_matrix)
from sklearn.preprocessing import LabelEncoder
import shap
import joblib
import os

USERNAME = "shreya"  # <- change to her username
BASE_PATH = f"/Users/{USERNAME}/Desktop/ARIA"
DATA_PATH = f"{BASE_PATH}/data"
MODEL_PATH = f"{BASE_PATH}/models"

print("✅ Imports done")

✅ Imports done


In [6]:
from sklearn.datasets import make_classification
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import joblib

np.random.seed(42)
n = 25000

# Generate realistic clinical features
df_hc = pd.DataFrame({
    'age': np.random.randint(18, 90, n),
    'time_in_hospital': np.random.randint(1, 14, n),
    'n_lab_procedures': np.random.randint(1, 100, n),
    'n_procedures': np.random.randint(0, 6, n),
    'n_medications': np.random.randint(1, 30, n),
    'n_outpatient': np.random.randint(0, 10, n),
    'n_inpatient': np.random.randint(0, 5, n),
    'n_emergency': np.random.randint(0, 5, n),
    'num_diagnoses': np.random.randint(1, 16, n),
    'has_diabetes': np.random.randint(0, 2, n),
    'has_hypertension': np.random.randint(0, 2, n),
    'has_heart_disease': np.random.randint(0, 2, n),
    'insulin_prescribed': np.random.randint(0, 2, n),
    'discharge_type': np.random.randint(1, 10, n),
    'admission_source': np.random.randint(1, 8, n),
})

# Create realistic target with clinical logic
readmit_prob = (
    0.15 +
    0.003 * df_hc['age'] +
    0.02 * df_hc['n_inpatient'] +
    0.01 * df_hc['n_emergency'] +
    0.05 * df_hc['has_diabetes'] +
    0.04 * df_hc['has_heart_disease'] +
    0.03 * df_hc['has_hypertension'] +
    0.002 * df_hc['n_medications'] -
    0.01 * df_hc['time_in_hospital'] +
    np.random.normal(0, 0.1, n)
)
readmit_prob = np.clip(readmit_prob, 0, 1)
df_hc['target'] = (readmit_prob > 0.5).astype(int)

print("Target distribution:")
print(df_hc['target'].value_counts())

# Features and target
features = [c for c in df_hc.columns if c != 'target']
X = df_hc[features]
y = df_hc['target']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train
model_hc = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    eval_metric='auc',
    random_state=42
)
model_hc.fit(X_train, y_train)

# Evaluate
y_pred_proba = model_hc.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"✅ Healthcare AUC-ROC: {auc:.4f}")
print(classification_report(y_test, model_hc.predict(X_test)))

# Save
joblib.dump(model_hc, f"{MODEL_PATH}/healthcare_model.pkl")
joblib.dump(features, f"{MODEL_PATH}/healthcare_features.pkl")
print("✅ Healthcare model saved")

Target distribution:
target
0    19918
1     5082
Name: count, dtype: int64
✅ Healthcare AUC-ROC: 0.8154
              precision    recall  f1-score   support

           0       0.85      0.94      0.89      3984
           1       0.61      0.34      0.44      1016

    accuracy                           0.82      5000
   macro avg       0.73      0.64      0.67      5000
weighted avg       0.80      0.82      0.80      5000

✅ Healthcare model saved


In [3]:
# Load
df = pd.read_csv(f"{DATA_PATH}/fintech_clean.csv")

# Features and target
X = df.drop('Class', axis=1)
y = df['Class']

# Handle class imbalance
fraud_count = y.sum()
legit_count = len(y) - fraud_count
scale = legit_count / fraud_count
print(f"Class imbalance ratio: {scale:.1f}x")

features_ft = X.columns.tolist()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train
model_ft = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale,
    eval_metric='auc',
    random_state=42
)
model_ft.fit(X_train, y_train)

# Evaluate
y_pred_proba = model_ft.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"✅ Fintech AUC-ROC: {auc:.4f}")
print(classification_report(y_test, model_ft.predict(X_test)))

# Save
joblib.dump(model_ft, f"{MODEL_PATH}/fintech_model.pkl")
joblib.dump(features_ft, f"{MODEL_PATH}/fintech_features.pkl")
print("✅ Fintech model saved")

Class imbalance ratio: 577.9x
✅ Fintech AUC-ROC: 0.9745
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.87      0.84      0.85        98

    accuracy                           1.00     56962
   macro avg       0.94      0.92      0.93     56962
weighted avg       1.00      1.00      1.00     56962

✅ Fintech model saved


In [4]:
# Load
df = pd.read_csv(f"{DATA_PATH}/saas_clean.csv")

# Encode target
df['target'] = (df['Churn'] == 'Yes').astype(int)

# Drop non-numeric and target columns
df = df.drop(['Churn', 'customerID'], axis=1)

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Encode categorical columns
cat_cols = df.select_dtypes(include='object').columns
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Features and target
X = df.drop('target', axis=1)
y = df['target']

features_saas = X.columns.tolist()

# Class imbalance
churn_count = y.sum()
retain_count = len(y) - churn_count
scale_saas = retain_count / churn_count
print(f"Class imbalance ratio: {scale_saas:.1f}x")

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train
model_saas = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_saas,
    eval_metric='auc',
    random_state=42
)
model_saas.fit(X_train, y_train)

# Evaluate
y_pred_proba = model_saas.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"✅ SaaS AUC-ROC: {auc:.4f}")
print(classification_report(y_test, model_saas.predict(X_test)))

# Save
joblib.dump(model_saas, f"{MODEL_PATH}/saas_model.pkl")
joblib.dump(features_saas, f"{MODEL_PATH}/saas_features.pkl")
print("✅ SaaS model saved")

/var/folders/nm/g335_yvn7pd518crzt0y87s40000gn/T/ipykernel_24661/80608966.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


Class imbalance ratio: 2.8x
✅ SaaS AUC-ROC: 0.8296
              precision    recall  f1-score   support

           0       0.88      0.77      0.82      1035
           1       0.53      0.72      0.61       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.79      0.76      0.77      1409

✅ SaaS model saved
